# Lab 04: Simulating Spiking Neurons
**ITAI-4374: Neuroscience as a Model for AI Systems**

**Student:** Win Aung  
**Date:** February 17, 2026

---

## Part 1: Implementing a Leaky Integrate-and-Fire (LIF) Neuron

The LIF model describes membrane potential dynamics:

**τ (dV/dt) = -(V - V_rest) + R · I**

When V reaches threshold V_th, the neuron fires and resets to V_reset.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
plt.rcParams['figure.figsize'] = [10, 5]
plt.rcParams['font.size'] = 11

In [2]:
class LIFNeuron:
    """
    Leaky Integrate-and-Fire (LIF) Neuron Model.
    
    The LIF model captures essential spiking dynamics:
    - Membrane potential integrates input current
    - Leak term pulls potential toward rest
    - Spike occurs when threshold is reached
    - Reset after spike with refractory period
    
    Reference: Gerstner & Kistler (2002), Spiking Neuron Models
    """
    
    def __init__(self, tau=20.0, V_rest=-70.0, V_th=-55.0, V_reset=-70.0, R=10.0, t_ref=2.0):
        """
        Initialize LIF neuron parameters.
        
        Args:
            tau: Membrane time constant (ms) - controls leak rate
            V_rest: Resting potential (mV) - stable voltage at rest
            V_th: Threshold potential (mV) - voltage to trigger spike
            V_reset: Reset potential after spike (mV)
            R: Membrane resistance (MOhm) - scales input current effect
            t_ref: Refractory period (ms) - minimum time between spikes
        """
        self.tau = tau
        self.V_rest = V_rest
        self.V_th = V_th
        self.V_reset = V_reset
        self.R = R
        self.t_ref = t_ref
        self.V = V_rest  # Current membrane potential
        self.t_since_spike = 1000  # Time since last spike (start high)
    
    def step(self, I, dt=0.1):
        """
        Simulate one time step using Euler integration.
        
        Args:
            I: Input current
            dt: Time step (ms)
            
        Returns:
            spike: 1 if neuron fired, 0 otherwise
        """
        spike = 0
        self.t_since_spike += dt
        
        # Only integrate if not in refractory period
        if self.t_since_spike > self.t_ref:
            # Euler integration: dV = (-(V - V_rest) + R*I) / tau * dt
            dV = (-(self.V - self.V_rest) + self.R * I) / self.tau * dt
            self.V += dV
            
            # Check for spike (all-or-none)
            if self.V >= self.V_th:
                spike = 1
                self.V = self.V_reset
                self.t_since_spike = 0
        
        return spike
    
    def reset(self):
        """Reset neuron to initial resting state."""
        self.V = self.V_rest
        self.t_since_spike = 1000
    
    def simulate(self, I_input, T, dt=0.1):
        """
        Run full simulation for duration T.
        
        Args:
            I_input: Input current (scalar or array matching time steps)
            T: Total simulation time (ms)
            dt: Time step (ms)
            
        Returns:
            t: Time array
            V: Membrane potential array
            spikes: List of spike times
        """
        self.reset()
        steps = int(T / dt)
        t = np.arange(0, T, dt)
        V = np.zeros(steps)
        spikes = []
        
        # Handle scalar vs array input
        if np.isscalar(I_input):
            I_input = np.ones(steps) * I_input
        
        for i in range(steps):
            V[i] = self.V
            spike = self.step(I_input[i], dt)
            if spike:
                spikes.append(t[i])
        
        return t, V, spikes

### Testing the LIF Neuron with Different Input Currents

In [3]:
# Create LIF neuron with standard parameters
neuron = LIFNeuron(tau=20, V_rest=-70, V_th=-55, V_reset=-70, R=10, t_ref=2)

print("=== LIF Neuron Response Tests ===")

# Test with three different input levels
fig, axes = plt.subplots(3, 1, figsize=(12, 10))
test_currents = [1.0, 2.0, 3.0]
labels = ['Subthreshold', 'At Threshold', 'Strong Input']
colors = ['blue', 'green', 'red']

for i, (I, label, c) in enumerate(zip(test_currents, labels, colors)):
    t, V, spikes = neuron.simulate(I, T=200, dt=0.1)
    
    axes[i].plot(t, V, color=c, linewidth=1.5, label=f'V(t)')
    axes[i].axhline(y=-55, color='red', linestyle='--', alpha=0.7, label='Threshold')
    axes[i].axhline(y=-70, color='gray', linestyle=':', alpha=0.5, label='V_rest')
    axes[i].set_ylabel('V (mV)')
    axes[i].set_title(f'{label}: I = {I}, {len(spikes)} spikes')
    axes[i].legend(loc='upper right')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_ylim(-80, -50)
    
    print(f"\nTest {i+1} - {label} (I={I}): {len(spikes)} spikes")
    if len(spikes) == 0:
        print("  Membrane potential stabilizes below threshold")
    else:
        print("  Neuron fires periodically" if I < 2.5 else "  Higher firing rate with stronger input")

axes[2].set_xlabel('Time (ms)')
plt.tight_layout()
plt.show()

=== LIF Neuron Response Tests ===

Test 1 - Subthreshold (I=1.0): 0 spikes
  Membrane potential stabilizes below threshold

Test 2 - At Threshold (I=2.0): 8 spikes
  Neuron fires periodically

Test 3 - Strong Input (I=3.0): 17 spikes
  Higher firing rate with stronger input


### Knowledge Check 1

**Q: What determines whether the neuron fires?**

**A:** The neuron fires when input current is strong enough to push the membrane potential above threshold (V_th = -55 mV) before the leak term pulls it back toward resting (V_rest = -70 mV). Key factors:

1. **Input current strength (I):** Higher I → faster rise toward threshold
2. **Membrane time constant (τ):** Smaller τ → faster leak, need stronger input
3. **Membrane resistance (R):** Higher R amplifies the current's effect
4. **Gap between V_rest and V_th:** Larger gap requires more current to overcome

---
## Part 2: Encoding Data as Spikes

Convert continuous signals to spike trains using different coding schemes.

In [4]:
def rate_encode(signal, max_rate=100, dt=1.0):
    """
    Rate coding: Higher signal values → higher spike probability.
    
    This is the most common encoding in biological neurons.
    Information is carried in the firing RATE over time.
    
    Args:
        signal: Input signal (normalized 0-1)
        max_rate: Maximum firing rate (Hz)
        dt: Time step (ms)
        
    Returns:
        spikes: Binary spike train (1 = spike, 0 = no spike)
    """
    # Probability of spike = rate * dt / 1000 (convert to ms)
    prob = signal * max_rate * dt / 1000
    prob = np.clip(prob, 0, 1)
    spikes = (np.random.random(len(signal)) < prob).astype(int)
    return spikes


def temporal_encode(signal, threshold=0.5):
    """
    Temporal coding: Spike when signal crosses threshold (rising edge).
    
    Information is carried in the precise TIMING of spikes.
    More efficient but sensitive to timing jitter.
    
    Args:
        signal: Input signal
        threshold: Crossing threshold
        
    Returns:
        spikes: Binary spike train
    """
    above = signal > threshold
    # Detect rising edges (transition from below to above)
    spikes = np.zeros(len(signal), dtype=int)
    spikes[1:] = (above[1:] & ~above[:-1]).astype(int)
    return spikes


def latency_encode(signal, max_latency=20):
    """
    Latency coding: Stronger signal → earlier spike.
    
    First spike time carries the information.
    Very energy efficient (one spike per stimulus).
    
    Args:
        signal: Input signal (normalized 0-1)
        max_latency: Maximum latency in time steps
        
    Returns:
        spikes: Binary spike train
    """
    # Higher signal = lower latency (earlier spike)
    latencies = ((1 - signal) * max_latency).astype(int)
    spikes = np.zeros(len(signal), dtype=int)
    
    # Place single spike at computed latency for each segment
    segment_size = len(signal) // 10
    for i in range(10):
        idx = i * segment_size + min(latencies[i * segment_size], segment_size - 1)
        spikes[idx] = 1
    
    return spikes

In [5]:
# Generate test signal: sinusoidal input
T = 500  # ms
dt = 1.0
t = np.arange(0, T, dt)
signal = 0.5 * (1 + np.sin(2 * np.pi * t / 100))  # Normalized 0-1

# Encode with different methods
rate_spikes = rate_encode(signal, max_rate=100, dt=dt)
temporal_spikes = temporal_encode(signal, threshold=0.5)

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

# Original signal
axes[0].plot(t, signal, 'b-', linewidth=2)
axes[0].set_ylabel('Signal')
axes[0].set_title('Original Signal (Sinusoidal)')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-0.1, 1.1)

# Rate encoding
spike_times_rate = t[rate_spikes == 1]
axes[1].eventplot(spike_times_rate, lineoffsets=0.5, colors='red', linewidths=1.5)
axes[1].set_ylabel('Spikes')
axes[1].set_title(f'Rate Encoding ({np.sum(rate_spikes)} spikes) - Density follows signal')
axes[1].set_ylim(0, 1)
axes[1].set_yticks([])

# Temporal encoding
spike_times_temp = t[temporal_spikes == 1]
axes[2].eventplot(spike_times_temp, lineoffsets=0.5, colors='green', linewidths=2)
axes[2].set_ylabel('Spikes')
axes[2].set_title(f'Temporal Encoding ({np.sum(temporal_spikes)} spikes) - Fires at threshold crossings')
axes[2].set_xlabel('Time (ms)')
axes[2].set_ylim(0, 1)
axes[2].set_yticks([])

plt.tight_layout()
plt.show()

print("=== Spike Encoding Comparison ===")
print(f"\nRate Encoding: {np.sum(rate_spikes)} spikes")
print("  - Spike density follows signal amplitude")
print("  - Stochastic (random given same input)")
print(f"\nTemporal Encoding: {np.sum(temporal_spikes)} spikes")
print("  - Fires only at threshold crossings")
print("  - Very sparse, precise timing")

=== Spike Encoding Comparison ===

Rate Encoding: 156 spikes
  - Spike density follows signal amplitude
  - Stochastic (random given same input)

Temporal Encoding: 5 spikes
  - Fires only at threshold crossings
  - Very sparse, precise timing


### Knowledge Check 2

**Q: What are the advantages of rate vs temporal coding?**

**A:**

| Aspect | Rate Coding | Temporal Coding |
|--------|-------------|----------------|
| **Speed** | Slow (needs time to estimate rate) | Fast (single spike carries info) |
| **Robustness** | Robust to noise | Sensitive to timing jitter |
| **Energy** | Many spikes needed | Efficient (few spikes) |
| **Decoding** | Count spikes over window | Detect precise timing |
| **Biology** | Motor cortex, slow signals | Auditory system, fast signals |

Biological neurons likely use **both** depending on the computational requirements.

---
## Part 3: Connecting a Small Spiking Network

In [6]:
class SpikingNetwork:
    """
    Simple feedforward spiking neural network.
    
    Architecture: Input → Hidden → Output
    Each layer consists of LIF neurons connected by synaptic weights.
    
    Spikes from one layer become weighted current input to the next.
    """
    
    def __init__(self, n_input, n_hidden, n_output):
        """
        Create network with specified architecture.
        
        Args:
            n_input: Number of input neurons
            n_hidden: Number of hidden layer neurons
            n_output: Number of output neurons
        """
        self.n_input = n_input
        self.n_hidden = n_hidden
        self.n_output = n_output
        
        # Create LIF neurons for each layer
        self.hidden = [LIFNeuron() for _ in range(n_hidden)]
        self.output = [LIFNeuron() for _ in range(n_output)]
        
        # Initialize synaptic weights (scaled by fan-in)
        self.W1 = np.random.randn(n_input, n_hidden) * 0.5 / np.sqrt(n_input)
        self.W2 = np.random.randn(n_hidden, n_output) * 0.5 / np.sqrt(n_hidden)
        
        # Synaptic scaling factor (converts spikes to current)
        self.syn_scale = 5.0
    
    def reset(self):
        """Reset all neurons to resting state."""
        for n in self.hidden + self.output:
            n.reset()
    
    def step(self, input_spikes, dt=0.1):
        """
        Process one time step through the network.
        
        Args:
            input_spikes: Binary array of input spikes
            dt: Time step
            
        Returns:
            hidden_spikes, output_spikes: Spike arrays for each layer
        """
        # Input → Hidden: Weight input spikes and feed to hidden neurons
        hidden_current = np.dot(input_spikes, self.W1) * self.syn_scale
        hidden_spikes = np.array([n.step(I, dt) for n, I in zip(self.hidden, hidden_current)])
        
        # Hidden → Output: Weight hidden spikes and feed to output neurons
        output_current = np.dot(hidden_spikes, self.W2) * self.syn_scale
        output_spikes = np.array([n.step(I, dt) for n, I in zip(self.output, output_current)])
        
        return hidden_spikes, output_spikes
    
    def simulate(self, input_spike_train, T, dt=0.1):
        """
        Run full simulation with input spike train.
        
        Args:
            input_spike_train: 2D array (time_steps x n_input)
            T: Simulation time (ms)
            dt: Time step
            
        Returns:
            hidden_history, output_history: Spike histories for each layer
        """
        self.reset()
        steps = int(T / dt)
        
        hidden_history = np.zeros((steps, self.n_hidden))
        output_history = np.zeros((steps, self.n_output))
        
        for i in range(steps):
            h_spikes, o_spikes = self.step(input_spike_train[i], dt)
            hidden_history[i] = h_spikes
            output_history[i] = o_spikes
        
        return hidden_history, output_history

In [7]:
# Create a small network
net = SpikingNetwork(n_input=3, n_hidden=5, n_output=2)

# Generate input spike trains with different rates
T = 200
dt = 0.1
steps = int(T / dt)
t = np.arange(0, T, dt)

# Three input channels with different firing rates
input_spikes = np.zeros((steps, 3))
for i in range(3):
    rate = 50 + i * 25  # Rates: 50, 75, 100 Hz
    prob = rate * dt / 1000
    input_spikes[:, i] = (np.random.random(steps) < prob).astype(int)

# Run simulation
hidden_hist, output_hist = net.simulate(input_spikes, T, dt)

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 9))
colors = ['C0', 'C1', 'C2', 'C3', 'C4']

# Input layer raster
for i in range(3):
    spike_times = t[input_spikes[:, i] == 1]
    axes[0].scatter(spike_times, np.ones_like(spike_times) * i, s=3, c=colors[i], label=f'Input {i+1}')
axes[0].set_ylabel('Input Neuron')
axes[0].set_title(f'Input Layer (3 neurons, {int(np.sum(input_spikes))} total spikes)')
axes[0].set_yticks([0, 1, 2])
axes[0].legend(loc='upper right', fontsize=8)
axes[0].grid(True, alpha=0.3)

# Hidden layer raster
for i in range(5):
    spike_times = t[hidden_hist[:, i] == 1]
    axes[1].scatter(spike_times, np.ones_like(spike_times) * i, s=5, c=colors[i])
axes[1].set_ylabel('Hidden Neuron')
axes[1].set_title(f'Hidden Layer (5 neurons, {int(np.sum(hidden_hist))} total spikes)')
axes[1].set_yticks(range(5))
axes[1].grid(True, alpha=0.3)

# Output layer raster
for i in range(2):
    spike_times = t[output_hist[:, i] == 1]
    axes[2].scatter(spike_times, np.ones_like(spike_times) * i, s=20, c=colors[i], marker='|')
axes[2].set_ylabel('Output Neuron')
axes[2].set_title(f'Output Layer (2 neurons, {int(np.sum(output_hist))} total spikes)')
axes[2].set_xlabel('Time (ms)')
axes[2].set_yticks([0, 1])
axes[2].set_ylim(-0.5, 1.5)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("=== Spiking Neural Network ===")
print(f"Architecture: {net.n_input} input → {net.n_hidden} hidden → {net.n_output} output")
print(f"\nSimulation Results ({T} ms):")
print(f"  Input spikes:  {int(np.sum(input_spikes))}")
print(f"  Hidden spikes: {int(np.sum(hidden_hist))}")
print(f"  Output spikes: {int(np.sum(output_hist))}")
print("\nNote: Information is compressed at each layer due to threshold mechanism.")

=== Spiking Neural Network ===
Architecture: 3 input → 5 hidden → 2 output

Simulation Results (200 ms):
  Input spikes:  298
  Hidden spikes: 84
  Output spikes: 11

Note: Information is compressed at each layer due to threshold mechanism.


### Knowledge Check 3

**Q: How does information flow through the spiking network?**

**A:** Information flows as follows:

1. **Input spikes arrive** and are weighted by synaptic connections (W1)
2. **Weighted spikes become current** input to hidden layer neurons
3. **Hidden neurons integrate** input and fire when reaching threshold
4. **Hidden spikes are weighted** by W2 and sent to output layer
5. **Output neurons integrate** hidden activity and produce final spikes

**Key observations:**
- The network **compresses information** (fewer spikes at each layer)
- **Coincident input spikes** are more effective (temporal summation)
- The **threshold mechanism** acts as a nonlinear filter
- **Timing matters** - spikes must arrive close together to sum effectively

---
## Part 4: Experiments

In [8]:
# Experiment 1: f-I Curve (Firing Rate vs Input Current)
print("\n=== Experiment 1: f-I Curve ===")

neuron = LIFNeuron()
currents = np.linspace(0, 4, 30)
firing_rates = []

for I in currents:
    _, _, spikes = neuron.simulate(I, T=500, dt=0.1)
    rate = len(spikes) / 0.5  # Convert to Hz (500 ms = 0.5 s)
    firing_rates.append(rate)

plt.figure(figsize=(10, 5))
plt.plot(currents, firing_rates, 'b-o', linewidth=2, markersize=5)
plt.xlabel('Input Current I', fontsize=12)
plt.ylabel('Firing Rate (Hz)', fontsize=12)
plt.title('f-I Curve: LIF Neuron (τ=20ms, R=10MΩ)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.axvline(x=1.5, color='r', linestyle='--', alpha=0.5, label='Rheobase ≈ 1.5')
plt.legend()
plt.tight_layout()
plt.show()

# Find rheobase (minimum current for firing)
rheobase_idx = np.argmax(np.array(firing_rates) > 0)
rheobase = currents[rheobase_idx]
print(f"Rheobase (minimum current to fire): ~{rheobase:.1f}")
print(f"Maximum firing rate: ~{max(firing_rates):.0f} Hz")
print("Note: Max rate limited by refractory period (2 ms → max ~500 Hz theoretical)")


=== Experiment 1: f-I Curve ===
Rheobase (minimum current to fire): ~1.5
Maximum firing rate: ~83 Hz
Note: Max rate limited by refractory period (2 ms → max ~500 Hz theoretical)


In [9]:
# Experiment 2: Effect of Time Constant τ
print("\n=== Experiment 2: Effect of Time Constant τ ===")

tau_values = [10, 20, 40]
colors = ['red', 'blue', 'green']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Time response comparison
for tau, c in zip(tau_values, colors):
    neuron = LIFNeuron(tau=tau)
    t, V, spikes = neuron.simulate(2.0, T=100, dt=0.1)
    axes[0].plot(t, V, color=c, linewidth=1.5, label=f'τ = {tau} ms ({len(spikes)} spikes)')

axes[0].axhline(y=-55, color='black', linestyle='--', alpha=0.5, label='Threshold')
axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('V (mV)')
axes[0].set_title('Membrane Potential Response (I = 2.0)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: f-I curves for different τ
for tau, c in zip(tau_values, colors):
    neuron = LIFNeuron(tau=tau)
    rates = []
    for I in currents:
        _, _, spikes = neuron.simulate(I, T=500, dt=0.1)
        rates.append(len(spikes) / 0.5)
    axes[1].plot(currents, rates, '-o', color=c, linewidth=2, markersize=3, label=f'τ = {tau} ms')

axes[1].set_xlabel('Input Current I')
axes[1].set_ylabel('Firing Rate (Hz)')
axes[1].set_title('f-I Curves for Different τ')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Findings:")
print("  τ = 10 ms: Fast response, higher firing rates, lower rheobase")
print("  τ = 20 ms: Standard response (baseline)")
print("  τ = 40 ms: Slow response, lower firing rates, higher rheobase")
print("\nInterpretation: τ controls the speed-accuracy trade-off.")
print("Small τ = fast but noisy; Large τ = slow but stable integration.")


=== Experiment 2: Effect of Time Constant τ ===

Key Findings:
  τ = 10 ms: Fast response, higher firing rates, lower rheobase
  τ = 20 ms: Standard response (baseline)
  τ = 40 ms: Slow response, lower firing rates, higher rheobase

Interpretation: τ controls the speed-accuracy trade-off.
Small τ = fast but noisy; Large τ = slow but stable integration.


In [10]:
# Experiment 3: Network response to different input patterns
print("\n=== Experiment 3: Network Response to Different Patterns ===")

net = SpikingNetwork(3, 5, 2)

T = 200
dt = 0.1
steps = int(T / dt)

# Pattern A: Bursting input (high frequency bursts)
pattern_A = np.zeros((steps, 3))
for burst_start in range(0, steps, 200):  # Burst every 20 ms
    burst_end = min(burst_start + 50, steps)  # 5 ms bursts
    pattern_A[burst_start:burst_end, :] = (np.random.random((burst_end - burst_start, 3)) < 0.3).astype(int)

# Pattern B: Steady low-rate input
pattern_B = (np.random.random((steps, 3)) < 0.05).astype(int)

# Simulate both patterns
hidden_A, output_A = net.simulate(pattern_A, T, dt)
net.reset()
hidden_B, output_B = net.simulate(pattern_B, T, dt)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Pattern A
im1 = axes[0, 0].imshow(pattern_A.T, aspect='auto', cmap='Blues', interpolation='nearest')
axes[0, 0].set_title(f'Pattern A: Bursting Input ({int(np.sum(pattern_A))} spikes)')
axes[0, 0].set_ylabel('Input Neuron')
axes[0, 0].set_yticks([0, 1, 2])

im2 = axes[0, 1].imshow(output_A.T, aspect='auto', cmap='Reds', interpolation='nearest')
axes[0, 1].set_title(f'Pattern A: Output ({int(np.sum(output_A))} spikes)')
axes[0, 1].set_ylabel('Output Neuron')
axes[0, 1].set_yticks([0, 1])

# Pattern B
im3 = axes[1, 0].imshow(pattern_B.T, aspect='auto', cmap='Blues', interpolation='nearest')
axes[1, 0].set_title(f'Pattern B: Steady Input ({int(np.sum(pattern_B))} spikes)')
axes[1, 0].set_ylabel('Input Neuron')
axes[1, 0].set_xlabel('Time Step')
axes[1, 0].set_yticks([0, 1, 2])

im4 = axes[1, 1].imshow(output_B.T, aspect='auto', cmap='Reds', interpolation='nearest')
axes[1, 1].set_title(f'Pattern B: Output ({int(np.sum(output_B))} spikes)')
axes[1, 1].set_ylabel('Output Neuron')
axes[1, 1].set_xlabel('Time Step')
axes[1, 1].set_yticks([0, 1])

plt.tight_layout()
plt.show()

print(f"\nPattern A (Bursting): Hidden={int(np.sum(hidden_A))} spikes, Output={int(np.sum(output_A))} spikes")
print("  - Strong bursts drive network effectively")
print("  - Output follows input structure")
print(f"\nPattern B (Steady): Hidden={int(np.sum(hidden_B))} spikes, Output={int(np.sum(output_B))} spikes")
print("  - Sparse input produces sparse output")
print("  - Less temporal structure preserved")


=== Experiment 3: Network Response to Different Patterns ===

Pattern A (Bursting): Hidden=156 spikes, Output=32 spikes
  - Strong bursts drive network effectively
  - Output follows input structure

Pattern B (Steady): Hidden=78 spikes, Output=9 spikes
  - Sparse input produces sparse output
  - Less temporal structure preserved


---
## Part 5: Reflection Questions

### Q1: How does the LIF model capture essential neural dynamics?

The LIF model captures the fundamental computational principles of biological neurons:

1. **Integration:** Accumulation of input over time (membrane potential sums weighted inputs)
2. **Leak:** Decay toward resting potential (forgetting old inputs, controlled by τ)
3. **Threshold:** Binary decision to fire or not (all-or-none principle)
4. **Reset:** Return to baseline after spike (ready for next computation)
5. **Refractory period:** Minimum time between spikes (limits maximum rate)

**What it simplifies away:**
- Detailed ion channel dynamics (Na+/K+ conductances)
- Realistic spike shape (action potential waveform)
- Adaptation and bursting patterns
- Dendritic computation

Despite simplifications, LIF captures the essence of neural computation: **integrate, threshold, fire**.

### Q2: What are advantages of SNNs over traditional ANNs?

| Advantage | Description |
|-----------|-------------|
| **Energy Efficiency** | Spikes are sparse; neurons only compute when active. Event-driven processing reduces power by 100-1000x on neuromorphic hardware. |
| **Temporal Coding** | Can encode information in spike timing, not just rate. Enables processing of temporal patterns (speech, motion). |
| **Event-Driven** | Natural fit for event cameras, audio, and real-time sensor data. No need to sample at fixed rate. |
| **Biological Realism** | Closer to brain computation. Enables brain-computer interfaces and neuroscience modeling. |
| **Neuromorphic Hardware** | Can run on specialized chips (Intel Loihi, IBM TrueNorth) with ultra-low power. |

**Challenges:**
- Harder to train (spikes are non-differentiable)
- Less mature software ecosystem
- Performance still behind ANNs on many benchmarks

### Q3: How might SNNs be applied in real AI systems?

**Promising applications:**

1. **Edge AI & IoT:** Ultra-low power processing for always-on sensors, wearables, embedded systems

2. **Robotics:** Real-time control using event cameras (DVS) for obstacle avoidance, tracking

3. **Neuromorphic Chips:** Intel Loihi for adaptive learning, IBM TrueNorth for pattern recognition

4. **Temporal Pattern Recognition:** Speech recognition, gesture detection, motion analysis

5. **Brain-Computer Interfaces:** Direct processing of neural signals from brain implants

6. **Autonomous Vehicles:** Fast, low-latency processing of sensor data for safety-critical decisions

**Key insight:** SNNs excel where **temporal patterns matter** and **energy is limited**.

### Q4: What did you learn from building the network simulation?

**Key insights from this lab:**

1. **Threshold creates sparsity:** Each layer produces fewer spikes than it receives, naturally compressing information.

2. **Timing is critical:** Coincident input spikes (arriving close together) are much more effective than spread-out spikes due to temporal summation.

3. **Parameters shape behavior:** τ, threshold, and synaptic weights completely determine whether a neuron acts as a fast coincidence detector or slow integrator.

4. **Information flows as events:** Unlike ANNs with continuous activations, SNNs communicate through discrete events (spikes), enabling asynchronous computation.

5. **Emergent complexity:** Simple neurons following simple rules (integrate, leak, fire) produce surprisingly complex network dynamics.

**Personal reflection:** Building these simulations deepened my understanding of how the brain might perform computation with such energy efficiency. The elegance of spike-based communication - where silence is free and only activity costs energy - is a powerful principle that AI systems are just beginning to exploit.

---
## References

1. Gerstner, W., & Kistler, W. M. (2002). *Spiking Neuron Models: Single Neurons, Populations, Plasticity*. Cambridge University Press.

2. Maass, W. (1997). Networks of spiking neurons: The third generation of neural network models. *Neural Networks, 10*(9), 1659-1671.

3. ITAI-4374 Module 04: Spiking Neural Networks and the LIF Model.

4. Pfeiffer, M., & Pfeil, T. (2018). Deep learning with spiking neurons. *Frontiers in Neuroscience, 12*, 774.